# NB_09_trial_demo - Trial Demo - Single Live Run

**Purpose.** Execute one TrialDefinition live in CARLA, with two ego configurations (base + a faster variant), an optional streaming-predictor variant, and a comparison video.

**Inputs.** a saved TrialDefinition under MIREIA/trials/, the perception checkpoints needed by the streaming predictor.

**Outputs.** per-run records under MIREIA/trials/<name>/runs/<timestamp>_<tag>/, including topdown + risk-overlay frames and a 2x2 dataset video.

**How to run.** Pick a trial in section 0b, then run the cells corresponding to the variants you want (baseline, streaming, fixed-route comparison).

**Position in the workflow.** Single-trial complement to the batch runners NB_10-13.


In [1]:
# Cell 1: all imports
import os
import random
import subprocess
import time
from pathlib import Path

import carla
import numpy as np

from MIREIA.analysis.visualizer import compose_trial_comparison_video, compose_trial_dataset_video
from MIREIA.config import Config
from MIREIA.data_collection.dataset_utils import load_jsonl_records
from MIREIA.perception import TemporalInferenceConfig, create_streaming_predictor
from MIREIA.simulation.bridge import SimulationBridge
from MIREIA.simulation.simple_route_controller import SimpleRouteController
from MIREIA.simulation.traffic_handler import TrafficHandler
from MIREIA.simulation.trials import EgoTrialConfig, TrialDefinition, TrialRunner
from MIREIA.simulation.world_manager import WorldManager

xFormers not available
xFormers not available


## 0) Global Configuration
Runtime parameters used by all sections (max steps, top-down resolution, CARLA launch flag). Trial-specific parameters (map, weather, route) come from a saved `TrialDefinition` loaded in the next section.

In [2]:
# Runtime parameters (independent of trial choice)
AUTO_LAUNCH_CARLA = True
CARLA_EXE = "CarlaUE4.exe"

MAX_STEPS_BASELINE = 6000
MAX_STEPS_STREAMING = 2000

TOPDOWN_RESOLUTION = (1024, 1024)
TOPDOWN_FOV = 95.0
TOPDOWN_ALIGN_RISK_ROTATION = True
TOPDOWN_CAPTURE_DELAY_TICKS = 2

# Manual dashboard crop used by the streaming predictor so its preprocessing
# matches the training pipeline (cuts the dashboard out of the 512x512 dashcam
# frame). Use trial_batch_runner.ipynb's preview cell to tune this bbox.
MANUAL_CROP_BBOX_XYXY: list[int] | None = [0, 0, 512, 305]

LEGACY_SCENARIO_NAME = "01A_ClearNoon_Town01_HighVol"

print("Runtime configuration loaded.")

Runtime configuration loaded.


## 0b) Select an Existing Trial
Lists all `TrialDefinition`s saved under `PATH_TO_TRIALS`. Use the dropdown to pick one (requires `ipywidgets`), then run the load cell. If the dropdown isn't available, set `TRIAL_NAME` manually in the load cell.

To create a new trial, use `trial_builder.ipynb`.

In [3]:
trials_root = Path(Config.PATH_TO_TRIALS)
existing_trials = sorted(p.parent.name for p in trials_root.glob("*/trial.json") if p.is_file())

trial_selector = None
if not existing_trials:
    print("No saved trials found. Run trial_builder.ipynb to create one.")
else:
    print(f"Found {len(existing_trials)} trial(s):")
    for name in existing_trials:
        print(f"  - {name}")

    try:
        import ipywidgets as widgets
        from IPython.display import display

        trial_selector = widgets.Dropdown(
            options=existing_trials,
            value=existing_trials[0],
            description="Trial:",
            layout=widgets.Layout(width="500px"),
        )
        display(trial_selector)
        print("\nPick a trial above, then run the next cell.")
    except ImportError:
        print("\nipywidgets not installed - set TRIAL_NAME manually in the next cell.")

Found 20 trial(s):
  - auto_17A_WetNoon_Town05_HighVol
  - auto_17B_WetNoon_Town05_LowVol
  - auto_17C_WetNoon_Town10HD_HighVol
  - auto_17D_WetNoon_Town10HD_LowVol
  - auto_18A_MidRainyNoon_Town05_HighVol
  - auto_18B_MidRainyNoon_Town05_LowVol
  - auto_18C_MidRainyNoon_Town10HD_HighVol
  - auto_18D_MidRainyNoon_Town10HD_LowVol
  - auto_19A_CloudySunset_Town05_HighVol
  - auto_19B_CloudySunset_Town05_LowVol
  - auto_19C_CloudySunset_Town10HD_HighVol
  - auto_19D_CloudySunset_Town10HD_LowVol
  - auto_20A_SoftRainSunset_Town05_HighVol
  - auto_20B_SoftRainSunset_Town05_LowVol
  - auto_20C_SoftRainSunset_Town10HD_HighVol
  - auto_20D_SoftRainSunset_Town10HD_LowVol
  - auto_21A_ClearNoon_Town05_HighVol_NoFog_Night
  - auto_21B_CloudyNoon_Town05_LowVol_NoFog_Night
  - auto_21C_WetNoon_Town10HD_HighVol_NoFog_Night
  - auto_21D_HardRainNoon_Town10HD_LowVol_NoFog_Night


Dropdown(description='Trial:', layout=Layout(width='500px'), options=('auto_17A_WetNoon_Town05_HighVol', 'auto…


Pick a trial above, then run the next cell.


In [4]:
# Load the selected trial. If the dropdown above is unavailable or you want a
# different one, set TRIAL_NAME by hand below.
TRIAL_NAME = trial_selector.value if trial_selector is not None else "demo_trial_runner"
# TRIAL_NAME = "my_trial_town03_clear"   # uncomment and edit to override

trial = TrialDefinition.load(TRIAL_NAME)

print(f"Loaded trial: {trial.name}")
print(f"  description: {trial.description}")
print(f"  map:         {trial.map_name}")
print(f"  weather:     {trial.weather}")
print(f"  vehicles:    {trial.n_vehicles}, pedestrians: {trial.n_pedestrians}")
print(f"  seed:        {trial.seed}")
print(f"  route_start: {trial.route_start}")
print(f"  route_end:   {trial.route_end}")

# Used by the advanced fixed-route sections (7-8) below.
START_CARLA = carla.Location(x=trial.route_start[0], y=trial.route_start[1], z=trial.route_start[2])
END_CARLA   = carla.Location(x=trial.route_end[0],   y=trial.route_end[1],   z=trial.route_end[2])

Loaded trial: auto_17A_WetNoon_Town05_HighVol
  description: Auto-generated trial from scenario 17A_WetNoon_Town05_HighVol.
  map:         Town05
  weather:     WetNoon
  vehicles:    80, pedestrians: 50
  seed:        42
  route_start: (-63.73694610595703, 6.2044758796691895, 0.0)
  route_end:   (207.2584228515625, 38.0629997253418, 0.0)


## 1) Optionally Launch CARLA
Skip this if CARLA is already running.

In [5]:
if AUTO_LAUNCH_CARLA:
    subprocess.Popen(CARLA_EXE, shell=True)
    print(f"Launched {CARLA_EXE}")
else:
    print("AUTO_LAUNCH_CARLA is False. Assuming CARLA is already running.")

Launched CarlaUE4.exe


## 2) Run Two Baseline Trials
Runs the trial loaded above with two ego configurations (base and faster speed) and records RGB / topdown / risk outputs.

In [6]:
ego_configs = [
    EgoTrialConfig(
        name="base_speed",
        ego_blueprint="vehicle.lincoln.mkz_2020",
        target_speed_kmh=20.0,
        speed_multiplier=1.0,
        use_vehicle_camera_defaults=True,
        controller_mode="behavior_agent",
        controller_behavior="normal",
    ),
    EgoTrialConfig(
        name="fast_speed",
        ego_blueprint="vehicle.lincoln.mkz_2020",
        target_speed_kmh=20.0,
        speed_multiplier=1.5,
        use_vehicle_camera_defaults=True,
        controller_mode="behavior_agent",
        controller_behavior="normal",
    ),
]
runner = TrialRunner(verbose=True)

In [7]:
summaries = runner.run_trial(
    trial=trial,
    ego_configs=ego_configs,
    max_steps=MAX_STEPS_BASELINE,
    image_stride=Config.RECORD_EVERY_N_TICKS,
    store_topdown_images=True,
    topdown_capture_delay_ticks=TOPDOWN_CAPTURE_DELAY_TICKS,
    topdown_resolution=TOPDOWN_RESOLUTION,
    topdown_fov=TOPDOWN_FOV,
    topdown_align_risk_rotation=TOPDOWN_ALIGN_RISK_ROTATION,
    store_risk_frame_images=True,
    store_static_risk_map=False,
    draw_debug_every_tick=True,
    draw_debug_skip_after_capture_ticks=1,
    draw_debug_skip_before_capture_ticks=0,
 )

print("\n=== Baseline Trial Summary ===")
for s in summaries:
    print(
        f"subtrial={s.subtrial_name} "
        f"frames={s.num_frames} "
        f"duration_s={s.duration_s:.2f} "
        f"distance_m={s.traveled_m:.2f} "
        f"risk_auc={s.risk_auc:.4f} "
        f"risk_per_meter={s.risk_per_meter:.6f} "
        f"finished={s.finished}"
    )
    print(f"summary_path={s.run_path}/summary.json")

Connecting to CARLA at 127.0.0.1:2000...



KeyboardInterrupt



## 3) Run Trial with Streaming Predictor
Loads a seq2seq checkpoint and runs one additional subtrial while logging predicted risk.

In [ ]:
stream_summaries = []
checkpoint_path = Path(Config.PATH_TO_MODELS) / "seq2seq_risk_checkpoint.pt"

if checkpoint_path.exists():
    streaming_predictor = create_streaming_predictor(
        checkpoint_path=str(checkpoint_path),
        temporal_config=TemporalInferenceConfig(
            sequence_len=Config.INFERENCE_SEQUENCE_LENGTH,
            burn_in_frames=Config.INFERENCE_BURN_IN_FRAMES,
            eval_frames=Config.INFERENCE_EVAL_FRAMES,
        ),
        strict=False,
        manual_crop_bbox=MANUAL_CROP_BBOX_XYXY,
    )

    trial_stream = TrialDefinition(
        name=f"{trial.name}_streaming",
        route_start=trial.route_start,
        route_end=trial.route_end,
        description="Streaming inference demo run.",
        map_name=trial.map_name,
        weather=trial.weather,
        n_vehicles=trial.n_vehicles,
        n_pedestrians=trial.n_pedestrians,
        seed=trial.seed,
        sync_mode=trial.sync_mode,
        fixed_delta=trial.fixed_delta,
    )

    stream_summaries = runner.run_trial(
        trial=trial_stream,
        ego_configs=[ego_configs[0]],
        max_steps=MAX_STEPS_STREAMING,
        image_stride=Config.RECORD_EVERY_N_TICKS,
        store_topdown_images=True,
        topdown_capture_delay_ticks=TOPDOWN_CAPTURE_DELAY_TICKS,
        topdown_resolution=TOPDOWN_RESOLUTION,
        topdown_fov=TOPDOWN_FOV,
        topdown_align_risk_rotation=TOPDOWN_ALIGN_RISK_ROTATION,
        store_risk_frame_images=True,
        streaming_predictor=streaming_predictor,
    )
    print("Streaming run created:", stream_summaries[0].run_path)
else:
    print(f"Skipping streaming demo: checkpoint not found at {checkpoint_path}")

## 4) Generate Comparison Video
Builds a 2x2 comparison video from the two baseline trial runs.

In [ ]:
if len(summaries) >= 2:
    jsonl_a = Path(summaries[0].run_path) / "dataset.jsonl"
    jsonl_b = Path(summaries[1].run_path) / "dataset.jsonl"

    comparison_video = compose_trial_comparison_video(
        jsonl_path_a=str(jsonl_a),
        jsonl_path_b=str(jsonl_b),
        output_name="trial_comparison_video.mp4",
        fps=Config.RECORDING_FPS,
        label_a=summaries[0].subtrial_name,
        label_b=summaries[1].subtrial_name,
        calculated_risk_key="predicted_risk",
    )
    print(f"Comparison video saved to: {comparison_video}")
else:
    print("Need at least 2 baseline summaries to run comparison video generation.")

## 5) Generate Single-Run Dataset Video
Creates a tiled dataset video (dashcam + risk/topdown + risk graph) for one selected run.

In [ ]:
selected_run = None
if stream_summaries:
    selected_run = Path(stream_summaries[0].run_path)
elif summaries:
    selected_run = Path(summaries[0].run_path)

if selected_run is None:
    print("No runs available yet. Run sections 2/3 first.")
else:
    jsonl_path = selected_run / "dataset.jsonl"
    if not jsonl_path.exists():
        raise FileNotFoundError(f"Dataset JSONL not found: {jsonl_path}")

    video_path = compose_trial_dataset_video(
        jsonl_path=str(jsonl_path),
        output_name="trial_dataset_video_with_risk_graph.mp4",
        fps=Config.RECORDING_FPS,
        calculated_risk_key="predicted_risk",
    )
    print(f"Saved dataset video to: {video_path}")

Saved dataset video to: d:\Projectes\TFG\MIREIA\trials\demo_trial_runner_streaming\runs\20260329_151057_base_speed\trial_dataset_video_with_risk_graph.mp4


## 5b) Speed Fusion Inference on Recorded Frames
Loads the speed-fusion checkpoint and predicts ego speed from consecutive frame pairs in one run JSONL.
It reports MAE/RMSE against `true_ego_speed` when labels are available.

In [ ]:
import math
from pathlib import Path

from MIREIA.perception import create_speed_fusion_predictor

speed_ckpt = Path(Config.PATH_TO_MODELS) / "speed_fusion_checkpoint.pt"
max_speed_pairs = 50

if selected_run is None:
    print("No run selected. Execute section 5 first.")
elif not speed_ckpt.exists():
    print(f"Speed-fusion checkpoint not found: {speed_ckpt}")
else:
    jsonl_path = selected_run / "dataset.jsonl"
    if not jsonl_path.exists():
        raise FileNotFoundError(f"Dataset JSONL not found: {jsonl_path}")

    records = load_jsonl_records(str(jsonl_path))
    if len(records) < 2:
        print("Not enough frames in dataset to run pairwise speed inference.")
    else:
        predictor = create_speed_fusion_predictor(checkpoint_path=str(speed_ckpt))

        def _resolve_image_path(record: dict, jsonl_dir: Path) -> Path | None:
            rel = record.get("rgb_image_path") or record.get("image_path")
            if not rel:
                return None
            p = Path(rel)
            if p.is_file():
                return p
            candidate = (jsonl_dir / p).resolve()
            if candidate.is_file():
                return candidate
            candidate = (Path(Config.PATH_TO_SCENARIOS) / p).resolve()
            if candidate.is_file():
                return candidate
            return None

        rows = []
        jsonl_dir = jsonl_path.parent
        processed = 0

        for i in range(len(records) - 1):
            if processed >= max_speed_pairs:
                break

            rec_a = records[i]
            rec_b = records[i + 1]
            img_a = _resolve_image_path(rec_a, jsonl_dir)
            img_b = _resolve_image_path(rec_b, jsonl_dir)
            if img_a is None or img_b is None:
                continue

            pred = predictor.predict(str(img_a), str(img_b), use_cache=True)

            gt = rec_b.get("true_ego_speed")
            if gt is None:
                ego = rec_b.get("ego", {})
                gt = ego.get("speed") if isinstance(ego, dict) else None
            gt = float(gt) if gt is not None else None

            rows.append({
                "frame_id": rec_b.get("frame_id", i + 1),
                "pred_mps": pred.speed_mps,
                "pred_kmh": pred.speed_kmh,
                "gt_mps": gt,
                "abs_err": abs(pred.speed_mps - gt) if gt is not None else None,
                "flow_mean": pred.flow_mean_magnitude,
                "flow_max": pred.flow_max_magnitude,
                "dyn_det": pred.dynamic_detection_count,
            })
            processed += 1

        if not rows:
            print("No valid frame pairs were found for speed inference.")
        else:
            print(f"Processed pairs: {len(rows)}")
            labeled_errors = [r["abs_err"] for r in rows if r["abs_err"] is not None]
            if labeled_errors:
                mae = sum(labeled_errors) / len(labeled_errors)
                rmse = math.sqrt(sum(e * e for e in labeled_errors) / len(labeled_errors))
                print(f"MAE (m/s):  {mae:.4f}")
                print(f"RMSE (m/s): {rmse:.4f}")

            preview = rows[:5]
            for item in preview:
                gt_text = f"{item['gt_mps']:.3f}" if item["gt_mps"] is not None else "n/a"
                err_text = f"{item['abs_err']:.3f}" if item["abs_err"] is not None else "n/a"
                print(
                    f"frame={item['frame_id']} pred={item['pred_mps']:.3f} m/s "
                    f"({item['pred_kmh']:.2f} km/h) gt={gt_text} err={err_text} "
                    f"flow_mean={item['flow_mean']:.4f} dyn={item['dyn_det']}"
                )

            speed_fusion_inference_rows = rows

## 6) Legacy JSONL Compatibility Check
Validates that a legacy scenario JSONL still contains required keys.

In [ ]:
legacy_jsonl = Path(Config.PATH_TO_SCENARIOS) / LEGACY_SCENARIO_NAME / "dataset.jsonl"
if legacy_jsonl.exists():
    legacy_records = load_jsonl_records(str(legacy_jsonl))
    first = legacy_records[0] if legacy_records else {}
    required_keys = ["frame_id", "rgb_image_path", "ground_truth_risk", "ego", "environment"]
    missing = [k for k in required_keys if k not in first]
    if missing:
        print("Legacy JSONL warning, missing keys:", missing)
    else:
        print("Legacy JSONL compatibility check passed.")
else:
    print(f"Legacy JSONL not found at {legacy_jsonl}")

## 7) Advanced: Direct Route Execution with Live Risk Sampling
Spawns ego manually, applies speed profile changes over time, and reports route risk AUC.

In [ ]:
client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()

prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = Config.SIM_FIXED_DELTA_SECONDS
world.apply_settings(settings)

spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points to run a route.")

ego_spawn_index = random.randrange(len(spawn_points))
handler = TrafficHandler(client, world, seed=SEED)
base_speed_kmh = 20.0
controller = SimpleRouteController(target_speed=base_speed_kmh, sampling_resolution=2.0)

def demo_speed_modifier(base_speed, tick_idx, _controller):
    phase = (tick_idx // 120) % 3
    if phase == 0:
        return 0.5 * base_speed
    if phase == 1:
        return 1.0 * base_speed
    return 2.0 * base_speed

controller.set_speed_modifier(demo_speed_modifier)
ego = handler.spawn_ego(
    blueprint_id="vehicle.lincoln.mkz_2020",
    spawn_index=ego_spawn_index,
    autopilot=False,
    controller=controller,
)

if hasattr(carla, "VehicleLightState"):
    light_state = (
        carla.VehicleLightState.Position
        | carla.VehicleLightState.LowBeam
        | carla.VehicleLightState.Interior
    )
    ego.set_light_state(carla.VehicleLightState(light_state))

world.tick()
start = ego.get_location()

candidates = []
for i, sp in enumerate(spawn_points):
    if i == ego_spawn_index:
        continue
    d = start.distance(sp.location)
    if d > 35.0:
        candidates.append((d, i))

if not candidates:
    raise RuntimeError("No suitable destination spawn point found.")

_, best_idx = random.choice(candidates)
end = spawn_points[best_idx].location
controller.set_destination(start, end)
print(f"Route set: start={start} → end={end}")

In [ ]:
bridge = SimulationBridge(world)
wm_risk = WorldManager(sync_mode=True, fixed_delta=Config.SIM_FIXED_DELTA_SECONDS, verbose=False)
wm_risk.world = world
wm_risk.bridge = bridge
wm_risk._WorldManager__bake_static_risk_map()
risk_values = []

for step in range(10000):
    handler.run_ego_controller_step()
    world.tick()
    bridge.update()
    try:
        risk_values.append(float(wm_risk.get_risk()))
    except RuntimeError:
        pass

    controller.draw_plan(world, max_points=1400, life_time=0.08)
    if controller.done():
        print(f"Route finished at step {step}")
        break

dt = settings.fixed_delta_seconds or Config.SIM_FIXED_DELTA_SECONDS
# np.trapz was removed in NumPy 2.0; pick whichever exists.
_trapezoid = getattr(np, "trapezoid", None) or np.trapz
if risk_values:
    total_risk_auc = float(_trapezoid(np.array(risk_values, dtype=np.float64), dx=dt))
    print(f"Total route risk AUC: {total_risk_auc:.4f}")
    print(f"Risk samples: {len(risk_values)} duration: {len(risk_values) * dt:.2f}s")
else:
    print("No risk samples collected.")

handler.destroy_all()
world.apply_settings(prev_settings)
print("Direct-route experiment done.")

## 8) Advanced: Fixed-Route Speed Comparison (No Traffic / With Traffic)
Compares risk-per-meter for the same route at 0.5x vs 2.0x speed, first without traffic, then with extra traffic.

In [ ]:
client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)
world = client.get_world()
world_map = world.get_map()
spawn_points = world_map.get_spawn_points()
if len(spawn_points) < 2:
    raise RuntimeError("Need at least 2 spawn points for fixed-route comparison.")

prev_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = Config.SIM_FIXED_DELTA_SECONDS
world.apply_settings(settings)

def as_location(value):
    if isinstance(value, carla.Transform):
        return value.location
    if isinstance(value, carla.Location):
        return value
    raise TypeError("START_CARLA and END_CARLA must be carla.Location or carla.Transform")

raw_start_loc = as_location(START_CARLA)
raw_end_loc = as_location(END_CARLA)

start_wp = world_map.get_waypoint(
    carla.Location(x=raw_start_loc.x, y=raw_start_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
)
end_wp = world_map.get_waypoint(
    carla.Location(x=raw_end_loc.x, y=raw_end_loc.y, z=0.0),
    project_to_road=True,
    lane_type=carla.LaneType.Driving,
)
if start_wp is None or end_wp is None:
    raise RuntimeError("Could not project START_CARLA/END_CARLA to a driving lane")

start_tf = start_wp.transform
start_tf.location.z += 0.5
start_loc = start_tf.location
end_loc = end_wp.transform.location

def nearest_spawn_indices(target_loc):
    dists = [(target_loc.distance(sp.location), i) for i, sp in enumerate(spawn_points)]
    dists.sort(key=lambda x: x[0])
    return [i for _, i in dists]

print(f"Route: start={start_loc} → end={end_loc}")

### `run_fixed_route` — Speed Comparison Helper

In [ ]:
def run_fixed_route(multiplier, base_speed_kmh=20.0, max_steps=10000, n_traffic=0):
    handler = TrafficHandler(client, world, seed=SEED)
    controller = SimpleRouteController(target_speed=base_speed_kmh, sampling_resolution=2.0)

    def speed_modifier(base_speed, tick_idx, _controller):
        return multiplier * base_speed

    controller.set_speed_modifier(speed_modifier)

    ego = None
    try:
        ego = handler.spawn_ego(
            blueprint_id="vehicle.lincoln.mkz_2020",
            spawn_point=start_tf,
            autopilot=False,
            controller=controller,
        )
    except RuntimeError as exc:
        print(f"Exact transform spawn failed: {exc}")

    if ego is None:
        for fallback_idx in nearest_spawn_indices(start_loc)[:20]:
            try:
                ego = handler.spawn_ego(
                    blueprint_id="vehicle.lincoln.mkz_2020",
                    spawn_index=fallback_idx,
                    autopilot=False,
                    controller=controller,
                )
                print(f"Spawn fallback succeeded at spawn_index={fallback_idx}")
                break
            except RuntimeError:
                continue

    if ego is None:
        raise RuntimeError("Failed to spawn ego vehicle: start position is blocked.")

    if n_traffic > 0:
        spawned_traffic = handler.spawn_vehicles(n=n_traffic, safe=True, car_lights_on=False)
        print(f"Spawned traffic vehicles: {len(spawned_traffic)}")

    if hasattr(carla, "VehicleLightState"):
        light_state = (
            carla.VehicleLightState.Position
            | carla.VehicleLightState.LowBeam
            | carla.VehicleLightState.Interior
        )
        ego.set_light_state(carla.VehicleLightState(light_state))

    world.tick()
    start = ego.get_location()
    controller.set_destination(start, end_loc)

    bridge = SimulationBridge(world)
    wm_risk = WorldManager(sync_mode=True, fixed_delta=Config.SIM_FIXED_DELTA_SECONDS, verbose=False)
    wm_risk.world = world
    wm_risk.bridge = bridge
    wm_risk._WorldManager__bake_static_risk_map()

    risks = []
    positions = []
    finished_step = None

    for step in range(max_steps):
        handler.run_ego_controller_step()
        world.tick()
        bridge.update()

        ego_kin = bridge.get_ego_kinematics()
        if ego_kin is not None:
            positions.append((float(ego_kin.x), float(ego_kin.y)))

        try:
            risks.append(float(wm_risk.get_risk()))
        except RuntimeError:
            pass

        controller.draw_plan(world, max_points=1400, life_time=0.08)
        if controller.done():
            finished_step = step
            break

    if len(positions) >= 2:
        pos_arr = np.array(positions, dtype=np.float64)
        seg = np.linalg.norm(np.diff(pos_arr, axis=0), axis=1)
        traveled_m = float(seg.sum())
    else:
        seg = np.array([], dtype=np.float64)
        traveled_m = 0.0

    risk_arr = np.array(risks, dtype=np.float64) if risks else np.array([], dtype=np.float64)
    if seg.size > 0 and risk_arr.size >= 2:
        n = min(seg.size, risk_arr.size - 1)
        risk_integral_distance = float(np.sum(0.5 * (risk_arr[:n] + risk_arr[1:n + 1]) * seg[:n]))
    else:
        risk_integral_distance = float("nan")

    risk_per_meter = (
        risk_integral_distance / traveled_m
        if traveled_m > 0 and np.isfinite(risk_integral_distance)
        else float("nan")
    )
    dt = settings.fixed_delta_seconds or Config.SIM_FIXED_DELTA_SECONDS

    handler.destroy_all()

    return {
        "multiplier": multiplier,
        "target_speed_kmh": multiplier * base_speed_kmh,
        "steps": finished_step if finished_step is not None else max_steps,
        "num_samples": len(risks),
        "duration_s": len(risks) * dt,
        "traveled_m": traveled_m,
        "risk_integral_distance": risk_integral_distance,
        "risk_per_meter": risk_per_meter,
    }

In [ ]:
print("=== No-traffic fixed-route comparison ===")
results = []
for m in (0.5, 2.0):
    res = run_fixed_route(m, base_speed_kmh=20.0, max_steps=10000, n_traffic=0)
    results.append(res)
    print(
        f"mult={res['multiplier']:.1f}x target={res['target_speed_kmh']:.1f} km/h "
        f"risk/m={res['risk_per_meter']:.6f} distance={res['traveled_m']:.2f}m "
        f"samples={res['num_samples']} duration={res['duration_s']:.2f}s"
    )

if len(results) == 2:
    a = results[0]["risk_per_meter"]
    b = results[1]["risk_per_meter"]
    if np.isfinite(a) and np.isfinite(b) and a != 0:
        print(f"risk/m ratio (2.0x / 0.5x): {b / a:.4f}")

In [ ]:
print("\n=== Extra-traffic fixed-route comparison ===")
results_traffic = []
for m in (0.5, 2.0):
    res = run_fixed_route(m, base_speed_kmh=20.0, max_steps=10000, n_traffic=45)
    results_traffic.append(res)
    print(
        f"[traffic] mult={res['multiplier']:.1f}x target={res['target_speed_kmh']:.1f} km/h "
        f"risk/m={res['risk_per_meter']:.6f} distance={res['traveled_m']:.2f}m "
        f"samples={res['num_samples']} duration={res['duration_s']:.2f}s"
    )

if len(results_traffic) == 2:
    a = results_traffic[0]["risk_per_meter"]
    b = results_traffic[1]["risk_per_meter"]
    if np.isfinite(a) and np.isfinite(b) and a != 0:
        print(f"[traffic] risk/m ratio (2.0x / 0.5x): {b / a:.4f}")

world.apply_settings(prev_settings)
print("Fixed-route comparisons done.")

## 5c) YOLOv11 ByteTrack Demo on Run Frames
Runs object tracking across consecutive RGB frames from the selected run JSONL.
Uses `track_objects(...)`, which internally calls YOLO `.track(..., persist=True, tracker="bytetrack.yaml")`.

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image
from ultralytics import YOLO

from MIREIA.perception.flow import track_objects

max_tracking_frames = 30
yolo_checkpoint = Path(Config.PATH_TO_MODELS) / "yolo11s.pt"

if selected_run is None:
    print("No run selected. Execute section 5 first.")
elif not yolo_checkpoint.exists():
    print(f"YOLO checkpoint not found: {yolo_checkpoint}")
else:
    jsonl_path = selected_run / "dataset.jsonl"
    if not jsonl_path.exists():
        raise FileNotFoundError(f"Dataset JSONL not found: {jsonl_path}")

    records = load_jsonl_records(str(jsonl_path))
    if not records:
        print("No records found in dataset JSONL.")
    else:
        model = YOLO(str(yolo_checkpoint))
        tracking_rows = []

        for idx, rec in enumerate(records[:max_tracking_frames]):
            rel = rec.get("rgb_image_path")
            if not rel:
                continue

            candidate = Path(rel)
            if not candidate.is_file():
                candidate = (jsonl_path.parent / rel).resolve()
            if not candidate.is_file():
                candidate = (Path(Config.PATH_TO_SCENARIOS) / rel).resolve()
            if not candidate.is_file():
                continue

            with Image.open(candidate) as img:
                frame_rgb = np.asarray(img.convert("RGB"), dtype=np.uint8)

            tracked = track_objects(model=model, frame_rgb=frame_rgb)
            tracking_rows.append(
                {
                    "frame_id": rec.get("frame_id", idx),
                    "image_path": str(candidate),
                    "tracked": tracked,
                }
            )

        print(f"Processed frames: {len(tracking_rows)}")
        if tracking_rows:
            print(f"First frame tracked objects: {len(tracking_rows[0]['tracked'])}")
            for item in tracking_rows[:3]:
                print(
                    f"frame={item['frame_id']} tracked={len(item['tracked'])} "
                    f"ids={[obj['id'] for obj in item['tracked'][:5]]}"
                )

        yolo_tracking_demo_rows = tracking_rows